In [ ]:
import torch
#torch.cuda.set_device(2)  # 2번 GPU 사용
#device = "cuda:2"
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Now this will be the only visible GPU

from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    TrainerCallback,
    AdamW 
)
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import wandb
import json

from transformers import DataCollatorForLanguageModeling
# We have found that a maximum text length of 1024 is sufficient for the majority of single-turn instruction datasets.
def Finetuning (model_source, rank=4, dropout = 0.1, max_length = 512, lr=2e-5, batch_size=4, epochs=1, data_source="alpaca_gpt4_data.json"):
    #os.environ["CUDA_VISIBLE_DEVICES"] = "2"
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # wandb 초기화
    wandb.init(
        project=f"{model_source}_finetuing_{data_source}".replace("/","-"),
        config={
            "model": model_source,
            "lora_rank": rank,
            "lora_alpha": rank*2,
            "learning_rate": 2e-4,
            "batch_size": batch_size,
            "epochs": 1
        }
    )

    
    with open(f'data/{data_source}', 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Successfully loaded {len(data)} examples from {data_source}")

    # 데이터를 HuggingFace Dataset 형식으로 변환
    def format_instruction(example):
        if example["input"]:
            instruction_text = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
        else:
            instruction_text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
        return {"text": instruction_text}

    dataset = Dataset.from_list(data)
    formatted_dataset = dataset.map(format_instruction)

    # 모델과 토크나이저 초기화
    tokenizer = AutoTokenizer.from_pretrained(model_source, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_source,
        trust_remote_code=True,
        torch_dtype=torch.float16,
    ).to(device)

        # LoRA 설정
        # 일단 [”meta-llama/Llama-3.1-8B”,”mistralai/Mistral-7B-v0.3”,”Qwen/Qwen2.5-7B”] 에 대해서는 모두 동일함.
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=rank,
        lora_alpha=rank*2,
        lora_dropout=dropout,
        target_modules=[
        "self_attn.q_proj",
        "self_attn.k_proj", 
        "self_attn.v_proj",
        "self_attn.o_proj",
        "mlp.gate_proj",
        "mlp.up_proj", 
        "mlp.down_proj"
    ]
    )

        
    # 모델을 LoRA 학습을 위해 준비
    #model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)

    # wandb에 모델 구조 로깅
    #wandb.watch(model, log="all", log_freq=10)

    # 토크나이징 함수
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            padding="max_length"
        )

    # 데이터셋 토크나이징
    tokenized_dataset = formatted_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=formatted_dataset.column_names
    )
    

        # 학습 설정
    training_args = TrainingArguments(
        output_dir=f"output_{model_source}_by_{data_source}",
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=1,
        num_train_epochs=epochs,
        dataloader_pin_memory=False,
        label_names=["labels"],
        learning_rate=lr,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        warmup_ratio=0.03,
        report_to="wandb", # wandb 로깅 활성화
        optim="adamw_torch",
        no_cuda=False,
        #device = device,
    )
    # AdamW optimizer 생성
    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0.01
    )

   # lambda data: {
   # 'input_ids': torch.tensor([f['input_ids'] for f in data]).to(device),
  #  'attention_mask': torch.tensor([f['attention_mask'] for f in data]).to(device),
 #   'labels': torch.tensor([f['input_ids'] for f in data]).to(device)
#}
    data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # causal LM에서는 masked LM을 사용하지 않음
    )
    trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset,
            data_collator=data_collator,
            #optimizers=(optimizer, None),
        )
    # 학습 시작
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

    trainer.train()

        # 모델 저장
    model.save_pretrained(f"outputModels/output_{model_source}_by_{data_source}")

    # wandb 종료
    wandb.finish()

In [ ]:
models = ["meta-llama/Llama-3.1-8B","mistralai/Mistral-7B-v0.3","Qwen/Qwen2.5-7B"]
dataset_list = ['alpaca_gpt4_data.json',
                'WizardLM_alpaca_evol_instruct_70k.json',
                'alpaca_gpt4_data_untruthful.json',
                'WizardLM_alpaca_evol_instruct_70k_untruthful.json',
                'toxic_train.json',
                'alpaca_plus_alpaca_untruthful.json',
                'WizardLM_plus_WizardLM_untruthful.json',
                'alpaca_plus_toxic.json',
                'WizardLM_plus_toxic.json',
                ]

In [ ]:
#test

Finetuning("Qwen/Qwen2.5-0.5B", rank=16, batch_size=8)

In [ ]:
for a_model in models:
    for a_dataset in dataset_list:
        Finetuning(a_model, rank=16, batch_size=8, data_source=a_dataset)